# 문항 1 아실 아파트 목록, 매물 크롤링

In [ ]:
# %pip install pandas openpyxl
# %pip install tqdm

In [1]:
import pandas as pd

data = pd.read_excel('법정동코드 조회자료.xlsx', sheet_name='Sheet0', header=0, dtype=str)
dong_data = data[data['법정동명'].str.split().str.len() == 3]
dong_data.reset_index(drop=True, inplace=True)

dong_code = dong_data['법정동코드'].tolist()

c:\Users\dyhan\anaconda3\envs\study\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
# collect_apt.py

import requests
import pandas as pd
import time

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
    logging.FileHandler("crawler.log", encoding="utf-8"),
    logging.StreamHandler(),
    ],
)

logger = logging.getLogger('crawler')

URL = 'https://asil.kr/app/data/data_apt_list.jsp'
HEADERS = {
    'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'referer': 'https://asil.kr/app/apt_list.jsp',
}
PARAMS = {
    'building' : '',
    'household' : '50',
    'order' : '0',
    'order_type' : '0',
}

time_now = time.strftime('%Y-%m-%d %H:%M:%S')
timeout = 10

class RetryException(Exception):
    pass
class StatusCodeException(Exception):
    pass

failed = []
skip = 0
def fetch(dong, timeout):
    for i in range(5):  # Retry up to 5 times
        global failed, skip, logger

        try:
            response = requests.get(URL, headers=HEADERS, params={**PARAMS, 'dong' : dong}, timeout=timeout)
            if response.status_code//100 == 5 or response.status_code == 429:
                # 재시도
                logger.warning(f"Encountered error {response.status_code} for dong {dong}. Retrying...")
                raise RetryException('재시도해야하는 오류')
            elif response.status_code in (400, 404) or response.status_code//100 == 4:
                raise requests.exceptions.RequestException
            timeout = 10  # Reset timeout after a successful request
            logger.info(f"Successfully fetched data for dong {dong}.")
            return response.json()
        except (TimeoutError, requests.exceptions.Timeout, requests.exceptions.ReadTimeout, RetryException) as e:
            print("Retrying")
            timeout *= 2  # Double the timeout for the next attempt
            logger.warning(f"Timeout or server error for dong {dong}. Retrying in {timeout} seconds...")
            continue  # Retry the request
        except requests.exceptions.RequestException as e:
            failed.append({
                '행정동' : dong,
                '에러' : str(e)
            })
            print("400 or 404 error")
            logger.error(f"Failed to fetch data for dong {dong}: {e}")
            timeout = 10
            break  # Break the loop for non-retryable errors
    timeout = 10
    logger.warning(f"Skipped dong {dong} after 5 retries.")
    skip += 1
    return {}

def parse(data):
    result = []
    for item in data:
        if len(item.get('offer', '')) == 0:
            offer = '매물 0건'
        else:
            offer = item.get('offer', '')
        result.append({
            'seq' : item.get('seq', ''),
            'status' : 'pending',
            'collected_at' : time_now,
            '행정동' : item.get('dongname', ''),
            '세대수' : item.get('household', '0'),
            '건축년도' : item.get('movein', ''),
            '아파트' : item.get('name', ''),
            '매물' : offer
        })
    return result

def collect_apt(dong_code):
    global failed, skip
    result = []

    for dong in dong_code:
        data = parse(fetch(dong, timeout))
        if data is not None:
            result.extend(data)

    print(f"성공: {len(result)}, 실패: {len(failed)}, 건너뜀: {skip}")
    logger.info(f"성공: {len(result)}, 실패: {len(failed)}, 건너뜀: {skip}")

    return result


data = collect_apt(dong_code)
df = pd.DataFrame(data)
df.to_csv('apts.csv', index=False, encoding='utf-8-sig')

df = pd.DataFrame(failed)
df.to_csv('failed_apt.csv', index=False, encoding='utf-8-sig')

2026-08-25 20:52:07,915 [INFO] Successfully fetched data for dong 1111010100.
2026-08-25 20:52:08,050 [INFO] Successfully fetched data for dong 1111010200.
2026-08-25 20:52:08,178 [INFO] Successfully fetched data for dong 1111010300.
2026-08-25 20:52:08,312 [INFO] Successfully fetched data for dong 1111010400.
2026-08-25 20:52:08,451 [INFO] Successfully fetched data for dong 1111010500.
2026-08-25 20:52:08,570 [INFO] Successfully fetched data for dong 1111010600.
2026-08-25 20:52:08,802 [INFO] Successfully fetched data for dong 1111010700.
2026-08-25 20:52:08,932 [INFO] Successfully fetched data for dong 1111010800.
2026-08-25 20:52:09,070 [INFO] Successfully fetched data for dong 1111010900.
2026-08-25 20:52:09,213 [INFO] Successfully fetched data for dong 1111011000.
2026-08-25 20:52:09,351 [INFO] Successfully fetched data for dong 1111011100.
2026-08-25 20:52:09,505 [INFO] Successfully fetched data for dong 1111011200.
2026-08-25 20:52:09,721 [INFO] Successfully fetched data for don

Retrying


2026-08-25 20:52:43,536 [INFO] Successfully fetched data for dong 1114016500.
2026-08-25 20:52:43,671 [INFO] Successfully fetched data for dong 1114016600.
2026-08-25 20:52:43,874 [INFO] Successfully fetched data for dong 1114016700.
2026-08-25 20:52:44,108 [INFO] Successfully fetched data for dong 1114016800.
2026-08-25 20:52:44,330 [INFO] Successfully fetched data for dong 1114016900.
2026-08-25 20:52:44,477 [INFO] Successfully fetched data for dong 1114017000.
2026-08-25 20:52:44,680 [INFO] Successfully fetched data for dong 1114017100.
2026-08-25 20:52:44,809 [INFO] Successfully fetched data for dong 1114017200.
2026-08-25 20:52:45,009 [INFO] Successfully fetched data for dong 1114017300.
2026-08-25 20:52:45,187 [INFO] Successfully fetched data for dong 1114017400.
2026-08-25 20:52:45,371 [INFO] Successfully fetched data for dong 1117010100.
2026-08-25 20:52:45,577 [INFO] Successfully fetched data for dong 1117010200.
2026-08-25 20:52:45,713 [INFO] Successfully fetched data for don

성공: 6793, 실패: 0, 건너뜀: 0


In [ ]:
# collect_forsale.py
# 10분+ 걸림

import pandas as pd
import requests

logger = logging.getLogger('crawler')

# 매물있는 아파트만
df = pd.read_csv('apts.csv', encoding='utf-8-sig')
df_apt = df[df['매물'] != '매물 0건']

URL = 'https://realty.asil.kr/api_asil/data_sale_of_apt_nomal.aspx'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'referer': 'https://asil.kr/app/apt_list.jsp'
}
FORM_DATA = {
    'oidx': 2,
    'oby': 'down',
    'total': 20,
}


timeout = 10

class RetryException(Exception):
    pass
class StatusCodeException(Exception):
    pass

failed_forsale = []
skip = 0

def fetch(seq, page=1):
    for i in range(5):  # Retry up to 5 times
        global failed_forsale, skip, logger, timeout

        try:
            response = requests.post(URL, data={**FORM_DATA, 'asil_bldcode': seq, 'focus_bldcode': seq, 'last_mm_num' : (page-1)*20}, headers=HEADERS, timeout=10)
            if response.status_code//100 == 5 or response.status_code == 429:
                # 재시도
                raise RetryException('재시도해야하는 오류')
            elif response.status_code in (400, 404) or response.status_code//100 == 4:
                raise requests.exceptions.RequestException
            timeout = 10  # Reset timeout after a successful request
            logger.info(f"Successfully fetched data for apartment {seq}, page {page}.")
            return response.json(), 0
        except (TimeoutError, requests.exceptions.Timeout, requests.exceptions.ReadTimeout, RetryException) as e:
            print("Retrying")
            timeout *= 2  # Double the timeout for the next attempt
            logger.warning(f"Timeout or server error for apartment {seq}. Retrying in {timeout} seconds...")
            continue  # Retry the request
        except requests.exceptions.RequestException as e:
            failed_forsale.append({
                '행정동' : seq,
                '에러' : str(e)
            })
            print("400 or 404 error")
            logger.error(f"Failed to fetch data for apartment {seq}: {e}")
            timeout = 10
            return {}, 1  # Break the loop for non-retryable errors
    timeout = 10
    skip += 1
    logger.error(f"Skipped apartment {seq} after 5 retries.")
    return {}, 2

def parse(data, code, seq):
    result = []
    if data['result'] == True:
        for item in data['list_result']:
            result.append({
                'seq' : seq,
                'uid' : item.get('mm_uid', ''),
                '중개사' : item.get('BRKG_NM', ''),
                '동' : item.get('BDONG_NM', ''),
                '층' : item.get('CORES_FLR_CNT_NM', ''),
                '매매가' : item.get('DEAL_AMT', ''),
                '보증금' : item.get('WRRNT_AMT', ''),
                '등록일' : item.get('SVC_DATE_STRT', ''),
                '월세' : item.get('LEASE_AMT', ''),
            })
        nxt_exists = data['next_page']
    else:
        failed_forsale.append({
            '행정동' : seq,
            '에러' : "No forsale data"
        })
        nxt_exists = False
    return result, nxt_exists, code

def collect_forsale(df):
    update = []
    result = []
    for seq in df['seq']:
        nxt_exists = True
        page = 1
        while nxt_exists:
            data, nxt_exists, code = parse(*fetch(seq, page), seq)
            result.extend(data)
            page += 1
        if code == 1 or code == 2:
            update.append({
                'seq' : seq,
                'status' : 'failed'
            })
        else:
            update.append({
                'seq' : seq,
                'status' : 'done'
            })

    print(f"성공: {len(result)}, 실패: {len(failed_forsale)}, 건너뜀: {skip}")
    logger.info(f"성공: {len(result)}, 실패: {len(failed_forsale)}, 건너뜀: {skip}")
    
    return result, update

data, update = collect_forsale(df_apt)
df = pd.DataFrame(data)
df.to_csv('collect_forsale.csv', index=False, encoding='utf-8-sig')

df = pd.DataFrame(failed_forsale)
df.to_csv('failed_forsale.csv', index=False, encoding='utf-8-sig')



성공: 76922, 실패: 2, 건너뜀: 0


In [17]:
import numpy as np

# update collect_apt.py
df_update = pd.DataFrame(update)
df_apt['status'] = np.array(df_update['status'])
df_apt.to_csv('apts.csv', index=False, encoding='utf-8-sig')